# 동영상 프레임 해설 파일 생성

## 0.환경 설정

In [1]:
base_path = r'C:\Users\playdata2\work_space(playdata)\SKN30_playdata\AI_agent\multimodal_ai'

In [2]:
%pip install openai pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import base64
import uuid
import re
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## 1.함수 정의

In [10]:
# Openai 응답이 HTML 오류일 경우 간결한 "API 서버 오류" 문구로 요약
def summarize_error_msg(description):
    if isinstance(description, str) and ('<!DOCTYPE html>' in description or '<html' in description.lower()):
        match = re.search(r'Error code (\d+)', description)
        if match:
            code = match.group(1)
            return f"API 서버 오류 (Error code {code})"
        return "API 서버 오류 (Unknown HTML)"
    return description

# 이미지 파일을 BASE64 문자열로 변환하여 vision 모델 입력 형태로 만드는 함수
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

def extract_frame_number(filename):
    m = re.search(r'_frame(\d+)\.jpg$', filename)
    return int(m.group(1)) if m else None


# 프레임 폴더명과 동일한 이름의 원본 영상 파일명을 videos_dir에서 찾는 함수
def get_video_filename_with_ext(video_name, videos_dir):
    for f in os.listdir(videos_dir):
        name, ext = os.path.splitext(f)
        if name == video_name:
            return f
    return video_name  # 혹시 못 찾으면 폴더명 그대로 반환

def call_openai_with_rate_limit(
    client,
    model,
    prompt_template,
    image_b64,
    max_retries=10,
    min_interval=1,
    show_error=False
):
    import time
    wait_time = min_interval
    last_call = getattr(call_openai_with_rate_limit, "last_call", None)
    if last_call:
        elapsed = time.time() - last_call
        if elapsed < min_interval:
            time.sleep(min_interval - elapsed)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model= model,
                messages=[
                    {"role": "system", "content": "당신은 이미지에 대해 정확하고 자세한 설명을 하는 장면 해설가입니다."},
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt_template },
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}}
                        ]
                    }
                ],
                max_tokens=256,
                temperature=0.3
            )
            call_openai_with_rate_limit.last_call = time.time()
            return response.choices[0].message.content
        except Exception as e:
            error_msg = str(e)
            reset_after = 0
            if hasattr(e, 'response') and hasattr(e.response, 'headers'):
                try:
                    reset_after = float(e.response.headers.get("x-ratelimit-reset-after", 0))
                except Exception:
                    reset_after = 0
            if "rate limit" in error_msg or "429" in error_msg or reset_after > 0:
                wait = max(wait_time, reset_after) if reset_after else wait_time
                if show_error:  # 에러 메시지를 보고 싶을 때만 출력
                    from tqdm import tqdm
                    tqdm.write(f"Rate limit 에러, {wait}초 대기 후 재시도 (시도 {attempt+1}/{max_retries})")
                time.sleep(wait)
                wait_time = min(wait_time * 2, 60)
                continue
            else:
                return f"Error: {error_msg}"
    return f"Error: {error_msg} (최대 재시도 {max_retries}회 초과)"


def is_error_description(description):
    if not isinstance(description, str):
        return True
    lower = description.lower()
    return lower.startswith("error") or "api 서버 오류" in lower or "<!doctype html" in lower or "<html" in lower

# 프레임 폴더를 순회하며 일정 간격으로 프레임을 vision 모델로 해설 생성 -> 성공/에러 csv 저장.
def generate_frame_descriptions(
    base_path='.',
    frames_dir='frames',
    videos_dir='video',
    video_names=None,
    frame_interval=1,
    model="gpt-4o-mini",
    prompt_template="이 이미지는 동영상의 한 프레임입니다. 프레임의 주요 특징이나 장면을 영어로 자세히 설명해 주세요.",
    openai_api_key=None,
    comments_dir='comments'
):
    # 실제 경로로 변환
    frames_dir = os.path.join(base_path, frames_dir)
    videos_dir = os.path.join(base_path, videos_dir)
    comments_dir = os.path.join(base_path, comments_dir)
    client = OpenAI(api_key=openai_api_key) if openai_api_key else OpenAI()
    os.makedirs(comments_dir, exist_ok=True)
    if video_names is None:
        video_names = [
            d for d in os.listdir(frames_dir)
            if os.path.isdir(os.path.join(frames_dir, d))
            and any(os.path.splitext(f)[0] == d for f in os.listdir(videos_dir))
        ]

    for video_name in video_names:
        video_filename = get_video_filename_with_ext(video_name, videos_dir)
        if not video_filename:
            print(f"[경고] videos 폴더에서 {video_name}.* 파일을 찾을 수 없습니다.")
        video_frame_dir = os.path.join(frames_dir, video_name)
        frame_files = sorted([f for f in os.listdir(video_frame_dir) if f.lower().endswith('.jpg')])
        results = []
        for idx, frame_file in enumerate(tqdm(frame_files, desc=f"{video_name} 프레임 해설 중")):
            if idx % frame_interval != 0:
                continue
            frame_path = os.path.join(video_frame_dir, frame_file)
            image_b64 = encode_image_to_base64(frame_path)
            frame_no = extract_frame_number(frame_file)
            description = call_openai_with_rate_limit(
                client, model, prompt_template, image_b64, max_retries=5, min_interval=1, show_error=False
            )
            description = summarize_error_msg(description)
            results.append({
                "id": str(uuid.uuid4()),
                "video_filename": video_filename,
                "frame_no": frame_no,
                "description": description
            })
        # 성공/실패 분리
        df = pd.DataFrame(results)
        df_success = df[~df['description'].apply(is_error_description)].reset_index(drop=True)
        df_error = df[df['description'].apply(is_error_description)].reset_index(drop=True)
        output_success = os.path.join(comments_dir, f"{video_name}_frames_success.csv")
        output_error = os.path.join(comments_dir, f"{video_name}_frames_error.csv")
        df_success.to_csv(output_success, index=False, encoding="utf-8-sig")
        df_error.to_csv(output_error, index=False, encoding="utf-8-sig")
        print(f"{video_name}: {len(df_success)}개 성공, {len(df_error)}개 에러 프레임 저장 완료.")

## 2.프레임 해설 파일 생성

In [11]:
generate_frame_descriptions(
    base_path=base_path,
    frames_dir='frames',
    videos_dir='video',
    video_names=None,
    frame_interval=30,          # 프레임 샘플링 간격, 30프레임마다 해설 생성
    model="gpt-4o",             # openai vision 모델
    comments_dir='comments'     # 결과 csv 저장 폴더
)

alpinist 프레임 해설 중: 100%|██████████| 208/208 [00:34<00:00,  6.10it/s]


alpinist: 7개 성공, 0개 에러 프레임 저장 완료.


basketball 프레임 해설 중: 100%|██████████| 172/172 [00:25<00:00,  6.75it/s]


basketball: 6개 성공, 0개 에러 프레임 저장 완료.


female_player_after_scoring 프레임 해설 중: 100%|██████████| 210/210 [00:28<00:00,  7.40it/s]


...

soccer_player_head: 7개 성공, 0개 에러 프레임 저장 완료.


surfer 프레임 해설 중: 100%|██████████| 195/195 [00:27<00:00,  6.98it/s]


surfer: 7개 성공, 0개 에러 프레임 저장 완료.


swimming 프레임 해설 중: 100%|██████████| 254/254 [00:34<00:00,  7.41it/s]

swimming: 9개 성공, 0개 에러 프레임 저장 완료.
